# Project: Customer Support System

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) LangGraph roadmap.

You will build a multi-agent customer support system with specialized subgraphs for billing, shipping, and returns. A router dispatches queries via conditional edges, persistence tracks conversation history, and a human approval gate protects refund actions.

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain "langchain[openai]"

## 2. Set Up Your API Key

Enter your OpenAI API key when prompted. The key is not stored or displayed.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Define State Schemas

Separate schemas for the parent graph and each department subgraph.

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import add_messages

class SupportState(TypedDict):
    messages: Annotated[list, add_messages]
    department: str
    customer_id: str
    resolution: str
    requires_approval: bool

class BillingState(TypedDict):
    messages: Annotated[list, add_messages]
    customer_id: str
    resolution: str

class ShippingState(TypedDict):
    messages: Annotated[list, add_messages]
    customer_id: str
    resolution: str

class ReturnsState(TypedDict):
    messages: Annotated[list, add_messages]
    customer_id: str
    resolution: str
    refund_amount: float
    requires_approval: bool

print("State schemas defined")
print(f"SupportState fields: {list(SupportState.__annotations__)}")
print(f"ReturnsState fields: {list(ReturnsState.__annotations__)}")

## 4. Build the Billing Subgraph

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

def billing_agent(state: BillingState) -> dict:
    response = llm.invoke([
        ("system",
         "You are a billing support agent. Help with invoices, payments, "
         "and subscriptions. Provide a clear resolution."),
        *state["messages"],
    ])
    return {
        "messages": [response],
        "resolution": response.content,
    }

billing_graph = StateGraph(BillingState)
billing_graph.add_node("agent", billing_agent)
billing_graph.add_edge(START, "agent")
billing_graph.add_edge("agent", END)
billing_app = billing_graph.compile()

print("Billing subgraph compiled")

## 5. Build the Shipping Subgraph

In [ ]:
def shipping_agent(state: ShippingState) -> dict:
    response = llm.invoke([
        ("system",
         "You are a shipping support agent. Help with tracking, delivery status, "
         "and address changes. Provide a clear resolution."),
        *state["messages"],
    ])
    return {
        "messages": [response],
        "resolution": response.content,
    }

shipping_graph = StateGraph(ShippingState)
shipping_graph.add_node("agent", shipping_agent)
shipping_graph.add_edge(START, "agent")
shipping_graph.add_edge("agent", END)
shipping_app = shipping_graph.compile()

print("Shipping subgraph compiled")

## 6. Build the Returns Subgraph with Human Approval

Refund requests route through a human approval gate.

In [ ]:
def returns_agent(state: ReturnsState) -> dict:
    response = llm.invoke([
        ("system",
         "You are a returns support agent. Process return requests. "
         "If the customer requests a refund, mention the refund in your response."),
        *state["messages"],
    ])
    needs_approval = any(
        word in response.content.lower()
        for word in ["refund", "credit", "reimburse"]
    )
    return {
        "messages": [response],
        "resolution": response.content,
        "requires_approval": needs_approval,
        "refund_amount": 50.00 if needs_approval else 0.0,
    }

def check_approval(state: ReturnsState) -> str:
    if state.get("requires_approval"):
        return "approval_gate"
    return "end"

def approval_gate(state: ReturnsState) -> dict:
    return {
        "resolution": (
            f"PENDING HUMAN APPROVAL: Refund of ${state['refund_amount']:.2f} "
            f"for customer {state['customer_id']}. "
            f"Original resolution: {state['resolution']}"
        ),
    }

returns_graph = StateGraph(ReturnsState)
returns_graph.add_node("agent", returns_agent)
returns_graph.add_node("approval_gate", approval_gate)
returns_graph.add_edge(START, "agent")
returns_graph.add_conditional_edges("agent", check_approval, {
    "approval_gate": "approval_gate",
    "end": END,
})
returns_graph.add_edge("approval_gate", END)
returns_app = returns_graph.compile()

print("Returns subgraph compiled (with approval gate)")

## 7. Build the Router and Parent Graph

In [ ]:
def classify_department(state: SupportState) -> dict:
    response = llm.invoke([
        ("system",
         "Classify the customer query into exactly one department: "
         "billing, shipping, or returns. Respond with only the department name."),
        *state["messages"],
    ])
    department = response.content.strip().lower()
    if department not in ("billing", "shipping", "returns"):
        department = "billing"
    return {"department": department}

def route_to_department(state: SupportState) -> str:
    return state["department"]

def handle_billing(state: SupportState) -> dict:
    result = billing_app.invoke({
        "messages": state["messages"],
        "customer_id": state["customer_id"],
    })
    return {
        "messages": result["messages"],
        "resolution": result["resolution"],
    }

def handle_shipping(state: SupportState) -> dict:
    result = shipping_app.invoke({
        "messages": state["messages"],
        "customer_id": state["customer_id"],
    })
    return {
        "messages": result["messages"],
        "resolution": result["resolution"],
    }

def handle_returns(state: SupportState) -> dict:
    result = returns_app.invoke({
        "messages": state["messages"],
        "customer_id": state["customer_id"],
    })
    return {
        "messages": result["messages"],
        "resolution": result["resolution"],
        "requires_approval": result.get("requires_approval", False),
    }

print("Router and handler nodes defined")

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

graph = StateGraph(SupportState)

graph.add_node("classify", classify_department)
graph.add_node("billing", handle_billing)
graph.add_node("shipping", handle_shipping)
graph.add_node("returns", handle_returns)

graph.add_edge(START, "classify")
graph.add_conditional_edges("classify", route_to_department, {
    "billing": "billing",
    "shipping": "shipping",
    "returns": "returns",
})
graph.add_edge("billing", END)
graph.add_edge("shipping", END)
graph.add_edge("returns", END)

checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

print("Customer support system compiled with persistence")

## 8. Test: Billing Query

In [ ]:
config = {"configurable": {"thread_id": "customer-001"}}

result = app.invoke(
    {
        "messages": [("human", "I was charged twice for my subscription last month")],
        "customer_id": "CUST-12345",
    },
    config=config,
)

print(f"Department: {result['department']}")
print(f"Resolution: {result['resolution'][:300]}...")

## 9. Test: Shipping Query

In [ ]:
config2 = {"configurable": {"thread_id": "customer-002"}}

result2 = app.invoke(
    {
        "messages": [("human", "Where is my package? Order #98765")],
        "customer_id": "CUST-67890",
    },
    config=config2,
)

print(f"Department: {result2['department']}")
print(f"Resolution: {result2['resolution'][:300]}...")

## 10. Test: Returns with Refund Approval

In [ ]:
config3 = {"configurable": {"thread_id": "customer-003"}}

result3 = app.invoke(
    {
        "messages": [("human", "I want to return a damaged item and get a refund")],
        "customer_id": "CUST-11111",
    },
    config=config3,
)

print(f"Department: {result3['department']}")
print(f"Requires approval: {result3.get('requires_approval')}")
print(f"Resolution: {result3['resolution'][:400]}...")

## 11. Test: Conversation Persistence

In [ ]:
result4 = app.invoke(
    {
        "messages": [("human", "Can you also check if I have any pending credits?")],
        "customer_id": "CUST-12345",
    },
    config=config,
)

print(f"Department: {result4['department']}")
print(f"Total messages in thread: {len(result4['messages'])}")
print(f"Resolution: {result4['resolution'][:300]}...")

## What You Learned

1. Each department is a **self-contained subgraph** with its own state schema
2. The **router** classifies queries with an LLM and dispatches via **conditional edges**
3. Subgraphs are called inside parent nodes (**Pattern 1**) since schemas differ
4. **Human-in-the-loop** approval gates protect high-value actions like refunds
5. **`MemorySaver`** enables persistent conversation history across turns
6. The pattern scales by adding new department subgraphs without modifying existing ones